<div style="border-left: 5px solid #1f4e79; padding: 16px 20px; background-color: #f8fafc; border: 1px solid #d9e2ec; border-radius: 6px; margin-bottom: 24px; font-family: Arial, sans-serif;">
  <h1 style="margin: 0 0 8px 0; color: #1f4e79; font-size: 28px;">08_01 — Démonstrateur de comparables immobiliers</h1>
  <p style="margin: 0; color: #4b5563; font-size: 15px; line-height: 1.55;">## Objectif<br><br>Ce notebook transforme la logique du notebook `08_00_construction_comparables_base_outil` en **démonstrateur métier**.<br><br>L&#x27;objectif n&#x27;est pas seulement de calculer une estimation, mais de montrer comment une approche experte par comparables peut être rendue :<br><br>? explicite ;<br>? paramétrable ;<br>? cartographiable ;<br>? contrôlable ;<br>? réutilisable dans un outil d&#x27;aide à la décision.<br><br>Le démonstrateur repose sur un cas de bien cible saisi par l&#x27;utilisateur : ville, type de bien, surface, pièces, date de référence et tolérances de comparaison.</p>
</div>


<div style="background-color: #f8fafc; border: 1px solid #d9e2ec; border-left: 4px solid #1f4e79; border-radius: 6px; padding: 14px 18px; margin: 16px 0; color: #1f2933; font-family: Arial, sans-serif; line-height: 1.55;">
  <h3 style="margin: 0 0 8px 0; color: #1f4e79; font-size: 18px;">1. Paramétrage des chemins</h3>
  <div style="color: #4b5563; font-size: 14px;">Adapte `PROJECT_ROOT` et `INPUT_PATH` à ton environnement local.<br><br>Le notebook attend idéalement une base issue de ton pipeline contenant au minimum :<br><br>? `id_mutation`<br>? `date_mutation`<br>? `prix_m2`<br>? `valeur_fonciere`<br>? `surface_reference`<br>? `nb_pieces_total`<br>? `type_bien`<br>? `nom_commune`<br>? `code_commune`<br>? `latitude`<br>? `longitude`<br><br>Si le fichier n&#x27;est pas disponible, un petit jeu de démonstration est généré automatiquement pour permettre de tester l&#x27;interface.</div>
</div>


In [1]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, HTML

try:
    import folium
    from folium.plugins import MarkerCluster
except ImportError:
    folium = None
    MarkerCluster = None
    print("Folium indisponible. Installez-le avec : pip install folium")

try:
    import ipywidgets as widgets
    from ipywidgets import interact, interactive_output, HBox, VBox
except ImportError:
    widgets = None
    print("ipywidgets indisponible. Installez-le avec : pip install ipywidgets")

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 80)

PROJECT_ROOT = Path(".")
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"

# À adapter avec ton vrai fichier
INPUT_PATH = PROCESSED_DIR / "df_biens_residentiels_comparables_light.parquet"

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "demo_comparables"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


<div style="background-color: #f8fafc; border: 1px solid #d9e2ec; border-left: 4px solid #1f4e79; border-radius: 6px; padding: 14px 18px; margin: 16px 0; color: #1f2933; font-family: Arial, sans-serif; line-height: 1.55;">
  <h3 style="margin: 0 0 8px 0; color: #1f4e79; font-size: 18px;">2. Chargement des données</h3>
  <div style="color: #4b5563; font-size: 14px;">Cette cellule charge la base réelle si elle existe. Sinon elle crée une base de démonstration permettant de valider le fonctionnement du notebook.</div>
</div>


In [2]:
def make_demo_base(n=8000, seed=42):
    rng = np.random.default_rng(seed)
    villes = {
        "Paris": (48.8566, 2.3522, 8500),
        "Lyon": (45.7640, 4.8357, 4700),
        "Toulouse": (43.6047, 1.4442, 3300),
        "Nantes": (47.2184, -1.5536, 3600),
        "Bordeaux": (44.8378, -0.5792, 4600),
        "Lille": (50.6292, 3.0573, 3600),
        "Rennes": (48.1173, -1.6778, 4100),
        "Le Mans": (48.0061, 0.1996, 2300),
    }
    rows = []
    for i in range(n):
        ville = rng.choice(list(villes.keys()))
        lat0, lon0, base_price = villes[ville]
        type_bien = rng.choice(["Appartement", "Maison"], p=[0.58, 0.42])
        surface = float(np.clip(rng.lognormal(4.25 if type_bien=="Appartement" else 4.55, 0.35), 20, 220))
        pieces = int(np.clip(round(surface / rng.normal(26, 6)), 1, 8))
        date = pd.Timestamp("2021-01-01") + pd.Timedelta(days=int(rng.integers(0, 5*365)))
        type_effect = 1.12 if type_bien == "Appartement" else 0.92
        surface_effect = 1 - np.clip((surface - 75) / 700, -0.12, 0.20)
        year_effect = {2021: .98, 2022: 1.04, 2023: 1.03, 2024: .99, 2025: 1.01}[date.year]
        prix_m2 = base_price * type_effect * surface_effect * year_effect + rng.normal(0, base_price * 0.18)
        prix_m2 = float(np.clip(prix_m2, 600, 14000))
        rows.append({
            "id_mutation": f"DEMO_{i:06d}",
            "date_mutation": date,
            "prix_m2": prix_m2,
            "valeur_fonciere": prix_m2 * surface,
            "surface_reference": surface,
            "nb_pieces_total": pieces,
            "type_bien": type_bien,
            "nom_commune": ville,
            "code_commune": f"{rng.integers(10000, 99999)}",
            "latitude": lat0 + rng.normal(0, 0.025),
            "longitude": lon0 + rng.normal(0, 0.025),
        })
    return pd.DataFrame(rows)

if INPUT_PATH.exists():
    df = pd.read_parquet(INPUT_PATH)
    print(f"Base réelle chargée : {INPUT_PATH}")
else:
    df = make_demo_base()
    print("Base de démonstration générée, faute de fichier réel.")

df["date_mutation"] = pd.to_datetime(df["date_mutation"], errors="coerce")
print(df.shape)
display(df.head())


Base de démonstration générée, faute de fichier réel.
(8000, 11)


,id_mutation,date_mutation,prix_m2,valeur_fonciere,surface_reference,nb_pieces_total,type_bien,nom_commune,code_commune,latitude,longitude
0,DEMO_000000,2024-11-13,6222.086721,567229.086497,91.163803,3,Appartement,Paris,57382,48.859796,2.344294
1,DEMO_000001,2021-11-30,2634.728104,137032.019256,52.009928,2,Appartement,Le Mans,93407,48.034281,0.211288
2,DEMO_000002,2022-02-19,4309.977881,216009.818290,50.118545,2,Appartement,Nantes,24870,47.201377,-1.523036
3,DEMO_000003,2022-10-27,4156.387394,338570.229760,81.457814,3,Maison,Rennes,52004,48.127618,-1.667029
4,DEMO_000004,2024-09-21,2337.585190,184903.389788,79.100172,4,Maison,Le Mans,81610,48.003251,0.178596


<div style="background-color: #f8fafc; border: 1px solid #d9e2ec; border-left: 4px solid #1f4e79; border-radius: 6px; padding: 14px 18px; margin: 16px 0; color: #1f2933; font-family: Arial, sans-serif; line-height: 1.55;">
  <h3 style="margin: 0 0 8px 0; color: #1f4e79; font-size: 18px;">3. Fonctions métier : distances, scoring, estimation</h3>
  <div style="color: #4b5563; font-size: 14px;">Le score combine quatre dimensions proches du raisonnement expert :<br><br>? proximité de surface ;<br>? proximité du nombre de pièces ;<br>? fraîcheur temporelle ;<br>? proximité géographique.<br><br>Les pondérations sont modifiables pour tester la sensibilité de l&#x27;estimation.</div>
</div>


In [3]:
def haversine_km(lat1, lon1, lat2, lon2):
    """Distance géographique approximative en kilomètres."""
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

def robust_minmax(s):
    s = pd.to_numeric(s, errors="coerce").replace([np.inf, -np.inf], np.nan)
    if s.notna().sum() == 0:
        return pd.Series(0.5, index=s.index)
    q05, q95 = s.quantile(0.05), s.quantile(0.95)
    if q05 == q95:
        return pd.Series(0.5, index=s.index)
    return ((s.clip(q05, q95) - q05) / (q95 - q05)).clip(0, 1)

def find_subject_coordinates(base, commune):
    sub = base[base["nom_commune"].eq(commune)]
    if sub.empty:
        return np.nan, np.nan
    return float(sub["latitude"].median()), float(sub["longitude"].median())

def build_comparables(
    base,
    commune,
    type_bien,
    surface_reference,
    nb_pieces_total,
    date_reference,
    rayon_km=2.0,
    mois_historique=36,
    tolerance_surface_pct=25,
    n_retenus=25,
    poids_surface=0.35,
    poids_pieces=0.20,
    poids_temps=0.25,
    poids_geo=0.20,
):
    """Construit un panel de comparables et une estimation."""
    data = base.copy()
    date_reference = pd.Timestamp(date_reference)
    lat_ref, lon_ref = find_subject_coordinates(data, commune)

    # Filtrage métier initial
    data = data[data["type_bien"].eq(type_bien)].copy()
    data = data[data["date_mutation"].notna()]
    data = data[data["date_mutation"].between(date_reference - pd.DateOffset(months=mois_historique), date_reference)]
    data = data[data["surface_reference"].between(
        surface_reference * (1 - tolerance_surface_pct/100),
        surface_reference * (1 + tolerance_surface_pct/100)
    )].copy()

    if np.isfinite(lat_ref) and np.isfinite(lon_ref):
        data["distance_km"] = haversine_km(lat_ref, lon_ref, data["latitude"], data["longitude"])
        data = data[data["distance_km"] <= rayon_km].copy()
    else:
        data["distance_km"] = np.nan

    if data.empty:
        return data, pd.DataFrame(), {"lat": lat_ref, "lon": lon_ref}

    # Distances analytiques
    data["distance_surface"] = (data["surface_reference"] - surface_reference).abs() / surface_reference
    data["distance_pieces"] = (data["nb_pieces_total"] - nb_pieces_total).abs()
    data["age_jours"] = (date_reference - data["date_mutation"]).dt.days.clip(lower=0)

    data["n_surface"] = robust_minmax(data["distance_surface"])
    data["n_pieces"] = robust_minmax(data["distance_pieces"])
    data["n_temps"] = robust_minmax(data["age_jours"])
    data["n_geo"] = robust_minmax(data["distance_km"])

    total_poids = poids_surface + poids_pieces + poids_temps + poids_geo
    data["score_global"] = (
        poids_surface * data["n_surface"]
        + poids_pieces * data["n_pieces"]
        + poids_temps * data["n_temps"]
        + poids_geo * data["n_geo"]
    ) / total_poids

    retenus = data.sort_values("score_global").head(n_retenus).copy()

    p = retenus["prix_m2"].dropna()
    estimation = pd.DataFrame([{
        "n_comparables": len(retenus),
        "prix_m2_q25": p.quantile(0.25),
        "prix_m2_mediane": p.median(),
        "prix_m2_q75": p.quantile(0.75),
        "estimation_basse": p.quantile(0.25) * surface_reference,
        "estimation_mediane": p.median() * surface_reference,
        "estimation_haute": p.quantile(0.75) * surface_reference,
        "dispersion_relative": (p.quantile(0.75) - p.quantile(0.25)) / p.median() if p.median() else np.nan,
        "score_median": retenus["score_global"].median(),
        "age_median_jours": retenus["age_jours"].median(),
        "distance_mediane_km": retenus["distance_km"].median(),
    }])

    return retenus, estimation, {"lat": lat_ref, "lon": lon_ref}


<div style="background-color: #f8fafc; border: 1px solid #d9e2ec; border-left: 4px solid #1f4e79; border-radius: 6px; padding: 14px 18px; margin: 16px 0; color: #1f2933; font-family: Arial, sans-serif; line-height: 1.55;">
  <h3 style="margin: 0 0 8px 0; color: #1f4e79; font-size: 18px;">4. Carte Folium des comparables</h3>
  <div style="color: #4b5563; font-size: 14px;">La carte localise le bien cible et les comparables retenus. Elle donne une lecture métier immédiate de la proximité géographique du panel.</div>
</div>


In [4]:
def build_map(comparables, subject_info, commune, surface, type_bien):
    if folium is None:
        print("Folium indisponible.")
        return None

    lat_ref, lon_ref = subject_info["lat"], subject_info["lon"]
    if not np.isfinite(lat_ref) or not np.isfinite(lon_ref):
        if comparables.empty:
            return None
        lat_ref, lon_ref = comparables["latitude"].median(), comparables["longitude"].median()

    m = folium.Map(location=[lat_ref, lon_ref], zoom_start=13, tiles="CartoDB positron")

    folium.Marker(
        [lat_ref, lon_ref],
        tooltip=f"Bien cible — {commune}",
        popup=f"<b>Bien cible</b><br>{type_bien}<br>{surface:.0f} m²",
        icon=folium.Icon(color="red", icon="home", prefix="fa")
    ).add_to(m)

    cluster = MarkerCluster(name="Comparables retenus").add_to(m) if MarkerCluster else m

    for _, r in comparables.iterrows():
        popup = (
            f"<b>Comparable</b><br>"
            f"Prix : {r['prix_m2']:.0f} €/m²<br>"
            f"Surface : {r['surface_reference']:.0f} m²<br>"
            f"Pièces : {r['nb_pieces_total']}<br>"
            f"Date : {pd.Timestamp(r['date_mutation']).date()}<br>"
            f"Score : {r['score_global']:.3f}<br>"
            f"Distance : {r['distance_km']:.2f} km"
        )
        folium.CircleMarker(
            location=[r["latitude"], r["longitude"]],
            radius=5,
            color="#2563eb",
            fill=True,
            fill_opacity=0.65,
            tooltip=f"{r['prix_m2']:.0f} €/m²",
            popup=popup
        ).add_to(cluster)

    folium.LayerControl().add_to(m)
    return m


<div style="background-color: #f8fafc; border: 1px solid #d9e2ec; border-left: 4px solid #1f4e79; border-radius: 6px; padding: 14px 18px; margin: 16px 0; color: #1f2933; font-family: Arial, sans-serif; line-height: 1.55;">
  <h3 style="margin: 0 0 8px 0; color: #1f4e79; font-size: 18px;">5. Démonstrateur interactif</h3>
  <div style="color: #4b5563; font-size: 14px;">Le formulaire permet de tester plusieurs biens cibles. Dans une soutenance, cette cellule peut servir de démonstration principale.</div>
</div>


In [5]:
def run_demo(
    commune,
    type_bien,
    surface_reference,
    nb_pieces_total,
    date_reference,
    rayon_km,
    mois_historique,
    tolerance_surface_pct,
    n_retenus
):
    comparables, estimation, subject_info = build_comparables(
        df,
        commune=commune,
        type_bien=type_bien,
        surface_reference=surface_reference,
        nb_pieces_total=nb_pieces_total,
        date_reference=date_reference,
        rayon_km=rayon_km,
        mois_historique=mois_historique,
        tolerance_surface_pct=tolerance_surface_pct,
        n_retenus=n_retenus,
    )

    display(HTML(f"<h3>Résultat — {commune} / {type_bien} / {surface_reference:.0f} m²</h3>"))

    if estimation.empty:
        display(HTML("<b style='color:red'>Aucun comparable trouvé avec ces critères. Élargir le rayon, la période ou la tolérance de surface.</b>"))
        return

    display(estimation.style.format({
        "prix_m2_q25": "{:,.0f} €/m²",
        "prix_m2_mediane": "{:,.0f} €/m²",
        "prix_m2_q75": "{:,.0f} €/m²",
        "estimation_basse": "{:,.0f} €",
        "estimation_mediane": "{:,.0f} €",
        "estimation_haute": "{:,.0f} €",
        "dispersion_relative": "{:.1%}",
        "score_median": "{:.3f}",
        "age_median_jours": "{:.0f}",
        "distance_mediane_km": "{:.2f}",
    }))

    display(HTML("<h4>Comparables retenus</h4>"))
    cols = ["date_mutation", "nom_commune", "type_bien", "surface_reference", "nb_pieces_total", "prix_m2", "distance_km", "score_global"]
    display(comparables[cols].head(n_retenus).style.format({
        "surface_reference": "{:.0f}",
        "prix_m2": "{:,.0f}",
        "distance_km": "{:.2f}",
        "score_global": "{:.3f}",
    }))

    # Graphique prix / score
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.scatter(comparables["score_global"], comparables["prix_m2"], alpha=0.7)
    ax.set_title("Prix au m² des comparables selon leur score")
    ax.set_xlabel("Score global de comparabilité, plus faible = meilleur")
    ax.set_ylabel("Prix au m²")
    plt.show()

    m = build_map(comparables, subject_info, commune, surface_reference, type_bien)
    if m is not None:
        display(m)

villes = sorted(df["nom_commune"].dropna().unique().tolist())
types = sorted(df["type_bien"].dropna().unique().tolist())

if widgets is not None:
    controls = {
        "commune": widgets.Dropdown(options=villes, value=villes[0], description="Commune"),
        "type_bien": widgets.Dropdown(options=types, value=types[0], description="Type"),
        "surface_reference": widgets.FloatSlider(value=70, min=20, max=220, step=5, description="Surface"),
        "nb_pieces_total": widgets.IntSlider(value=3, min=1, max=8, step=1, description="Pièces"),
        "date_reference": widgets.DatePicker(value=pd.Timestamp("2025-12-31").date(), description="Date"),
        "rayon_km": widgets.FloatSlider(value=2.0, min=0.3, max=10.0, step=0.1, description="Rayon km"),
        "mois_historique": widgets.IntSlider(value=36, min=6, max=60, step=6, description="Mois"),
        "tolerance_surface_pct": widgets.IntSlider(value=25, min=5, max=60, step=5, description="Tolérance %"),
        "n_retenus": widgets.IntSlider(value=25, min=5, max=80, step=5, description="N retenus"),
    }
    ui = VBox([
        HBox([controls["commune"], controls["type_bien"], controls["date_reference"]]),
        HBox([controls["surface_reference"], controls["nb_pieces_total"]]),
        HBox([controls["rayon_km"], controls["mois_historique"], controls["tolerance_surface_pct"], controls["n_retenus"]]),
    ])
    out = interactive_output(run_demo, controls)
    display(ui, out)
else:
    run_demo(
        commune=villes[0],
        type_bien=types[0],
        surface_reference=70,
        nb_pieces_total=3,
        date_reference="2025-12-31",
        rayon_km=2.0,
        mois_historique=36,
        tolerance_surface_pct=25,
        n_retenus=25,
    )


Output()

<div style="background-color: #f8fafc; border: 1px solid #d9e2ec; border-left: 4px solid #1f4e79; border-radius: 6px; padding: 14px 18px; margin: 16px 0; color: #1f2933; font-family: Arial, sans-serif; line-height: 1.55;">
  <h3 style="margin: 0 0 8px 0; color: #1f4e79; font-size: 18px;">6. Export des résultats</h3>
  <div style="color: #4b5563; font-size: 14px;">Cette cellule permet d&#x27;exporter un panel de comparables et une estimation pour un cas donné.</div>
</div>


In [6]:
# Exemple d'export reproductible
comparables_export, estimation_export, subject_info_export = build_comparables(
    df,
    commune=sorted(df["nom_commune"].dropna().unique())[0],
    type_bien=sorted(df["type_bien"].dropna().unique())[0],
    surface_reference=70,
    nb_pieces_total=3,
    date_reference="2025-12-31",
    rayon_km=2.0,
    mois_historique=36,
    tolerance_surface_pct=25,
    n_retenus=25,
)

comparables_export.to_csv(OUTPUT_DIR / "comparables_retenus_demo.csv", index=False)
estimation_export.to_csv(OUTPUT_DIR / "estimation_demo.csv", index=False)

print("Exports créés dans :", OUTPUT_DIR.resolve())


Exports créés dans : C:\Users\club_\OneDrive\13_DOCUMENT\SYSTEME_AIDE_DECISION_IMMOBILIERE\notebooks\outputs\demo_comparables


<div style="background-color: #f8fafc; border: 1px solid #d9e2ec; border-left: 4px solid #1f4e79; border-radius: 6px; padding: 14px 18px; margin: 16px 0; color: #1f2933; font-family: Arial, sans-serif; line-height: 1.55;">
  <h3 style="margin: 0 0 8px 0; color: #1f4e79; font-size: 18px;">7. Lecture métier</h3>
  <div style="color: #4b5563; font-size: 14px;">Le démonstrateur illustre trois principes importants :<br><br>1. **La valeur est contextualisée** : elle dépend d&#x27;un marché local, pas seulement d&#x27;un bien.<br>2. **Le comparable est une information de marché** : il apporte une mémoire récente des transactions.<br>3. **L&#x27;estimation doit rester explicable** : nombre de comparables, distance, fraîcheur et dispersion doivent être visibles.<br><br>Dans une version opérationnelle, cette brique peut alimenter :<br><br>? un observatoire immobilier ;<br>? une API de valorisation ;<br>? un outil Streamlit ;<br>? une analyse de garanties bancaires ;<br>? un module d&#x27;aide à la décision.</div>
</div>
